# 01 — Data Collection & Initial Inspection

This notebook documents the raw INMET weather data files:
- what we received
- what each file looks like
- any immediate issues spotted before cleaning

No data is modified here. All files read from `data/raw/`.

In [1]:
import pandas as pd
from pathlib import Path

# Project root — works regardless of where the notebook is called from
ROOT = Path().resolve()
RAW = ROOT / "data" / "raw"

print(ROOT)
print(RAW)
print("Exists:", RAW.exists())

D:\projects\air_quality_brazil
D:\projects\air_quality_brazil\data\raw
Exists: True


In [2]:
# List all files in raw/
files = list(RAW.glob("**/*"))
for f in files:
    print(f.relative_to(ROOT))

data\raw\hotspots
data\raw\INMET
data\raw\SISAM
data\raw\hotspots\2022
data\raw\hotspots\2023
data\raw\hotspots\2024
data\raw\hotspots\2025
data\raw\INMET\2022
data\raw\INMET\2023
data\raw\INMET\2024
data\raw\INMET\2025
data\raw\SISAM\AM sisam_municipio_reanalise_diario_2022-01-01_2025-12-31.csv
data\raw\SISAM\BH sisam_municipio_reanalise_diario_2022-01-01_2025-12-31.csv
data\raw\SISAM\DF sisam_municipio_reanalise_diario_2022-01-01_2025-12-31.csv
data\raw\SISAM\PA sisam_municipio_reanalise_diario_2022-01-01_2025-12-31.csv
data\raw\SISAM\SP sisam_municipio_reanalise_diario_2022-01-01_2025-12-31.csv
data\raw\hotspots\2022\AM exportador_2025-09-24 12_37_02.952865.csv
data\raw\hotspots\2022\DF bdqueimadas_2022-01-01_2022-12-31.csv
data\raw\hotspots\2022\MG exportador_2025-10-21 21_39_16.068456.csv
data\raw\hotspots\2022\RS bdqueimadas_2022-01-01_2022-12-31.csv
data\raw\hotspots\2022\SP bdqueimadas_2022-01-01_2022-12-31.csv
data\raw\hotspots\2023\AM exportador_2025-09-24 12_36_33.646863.csv

## Things to note about data origin

* Porto Alegre's second station changes code in 2025 (B807 -> B825)
* Belo Horizonte gets a new station from 28-08-2025
* B807 starts at the end of 2022, only 3 weeks of data

In [3]:
# Pick one file to inspect — we'll use this to understand the raw format before writing any general reader function
sample_path = RAW / "INMET" / "2022" / "INMET_SE_SP_A701_SAO PAULO - MIRANTE_01-01-2022_A_31-12-2022.CSV"

# Read the first 15 lines as raw text — no parsing yet
# This shows us the header structure before pandas touches it
with open(sample_path, encoding="latin-1") as f:
    for i, line in enumerate(f):
        print(f"{i:02d}: {line}", end="")
        if i == 14:
            break

00: REGIAO:;SE
01: UF:;SP
02: ESTACAO:;SAO PAULO - MIRANTE
03: CODIGO (WMO):;A701
04: LATITUDE:;-23,49638888
05: LONGITUDE:;-46,61999999
06: ALTITUDE:;785,64
07: DATA DE FUNDACAO:;25/07/06
08: Data;Hora UTC;PRECIPITAÇÃO TOTAL, HORÁRIO (mm);PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB);PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB);PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB);RADIACAO GLOBAL (Kj/m²);TEMPERATURA DO AR - BULBO SECO, HORARIA (°C);TEMPERATURA DO PONTO DE ORVALHO (°C);TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C);TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C);TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C);TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C);UMIDADE REL. MAX. NA HORA ANT. (AUT) (%);UMIDADE REL. MIN. NA HORA ANT. (AUT) (%);UMIDADE RELATIVA DO AR, HORARIA (%);VENTO, DIREÇÃO HORARIA (gr) (° (gr));VENTO, RAJADA MAXIMA (m/s);VENTO, VELOCIDADE HORARIA (m/s);
09: 2022/01/01;0000 UTC;0;920,5;920,5;920,1;;19,3;18;19,4;19,2;18,2;17,8;92;92;92;35;2,1;,7;
10: 2022/01

## Things to note about file structure
* 8 rows of metadata before the column headers (row 08) — so skiprows=8 when reading with pandas
* Separator is ; not ,
* Decimal separator is , not . — pandas has a decimal parameter for this
* Encoding is latin-1 as expected
* The time column says 0000 UTC, 0100 UTC etc — we'll need to clean that to get proper datetimes
* There's a trailing ; at the end of every row — pandas will create an extra empty column, we'll drop it
* Some values are missing entirely (see row 09, the RADIACAO GLOBAL column is blank)*

In [4]:
def read_inmet_file(filepath):
    """
    Reads a single INMET automatic station CSV file and returns a clean DataFrame.

    INMET files have 8 rows of station metadata before the actual column headers,
    use semicolon as separator, comma as decimal, and latin-1 encoding.
    """

    # Read the metadata rows separately so we can extract station info
    # This gives us city, station code, coordinates etc from the file itself
    with open(filepath, encoding="latin-1") as f:
        meta_lines = [next(f).strip() for _ in range(8)]

    # Parse metadata into a dict — each line is "KEY:;VALUE"
    meta = {}
    meta_keys = ["regiao", "uf", "estacao", "codigo_wmo", "latitude", "longitude", "altitude", "data_fundacao"]
    for key, line in zip(meta_keys, meta_lines):
        # Split on ;, take the second element, strip whitespace
        meta[key] = line.split(";")[1].strip() if ";" in line else ""

    # Read the actual data, skipping the 8 metadata rows
    df = pd.read_csv(
        filepath,
        sep=";",           # INMET uses semicolons
        skiprows=8,        # skip the metadata header
        decimal=",",       # Brazilian decimal format uses comma
        encoding="latin-1",
        low_memory=False,  # avoids dtype guessing warnings on large files
    )

    # Drop the last column — trailing semicolons create an empty unnamed column
    df = df.drop(columns=df.columns[-1])

    # Attach station metadata as columns so we know where each row came from after we eventually concatenate all files together
    df["estacao"] = meta["estacao"]
    df["codigo_wmo"] = meta["codigo_wmo"]
    df["uf"] = meta["uf"]
    df["latitude"] = meta["latitude"]
    df["longitude"] = meta["longitude"]

    return df, meta


# Test it on our sample file
df_sample, meta_sample = read_inmet_file(sample_path)

print("Metadata extracted:")
for k, v in meta_sample.items():
    print(f"  {k}: {v}")

print(f"\nDataFrame shape: {df_sample.shape}")
print(f"\nColumns:\n{df_sample.columns.tolist()}")
print(f"\nFirst 3 rows:")
df_sample.head(3)

Metadata extracted:
  regiao: SE
  uf: SP
  estacao: SAO PAULO - MIRANTE
  codigo_wmo: A701
  latitude: -23,49638888
  longitude: -46,61999999
  altitude: 785,64
  data_fundacao: 25/07/06

DataFrame shape: (8760, 24)

Columns:
['Data', 'Hora UTC', 'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)', 'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)', 'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)', 'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)', 'RADIACAO GLOBAL (Kj/m²)', 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)', 'TEMPERATURA DO PONTO DE ORVALHO (°C)', 'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)', 'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)', 'TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)', 'TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)', 'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)', 'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)', 'UMIDADE RELATIVA DO AR, HORARIA (%)', 'VENTO, DIREÇÃO HORARIA (gr) (° (gr))', 'VENTO, RAJADA MAXIMA (m/s)', 'VENTO, VELOCIDADE HORARIA (m/s)', 'estacao', 'c

,Data,Hora UTC,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)","PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (Kj/m²),"TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",TEMPERATURA DO PONTO DE ORVALHO (°C),TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C),...,UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIREÇÃO HORARIA (gr) (° (gr))","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",estacao,codigo_wmo,uf,latitude,longitude
0,2022/01/01,0000 UTC,0.0,920.5,920.5,920.1,NaN,19.3,18.0,19.4,...,92,92,35,2.1,0.7,SAO PAULO - MIRANTE,A701,SP,"-23,49638888","-46,61999999"
1,2022/01/01,0100 UTC,0.0,920.8,920.9,920.5,NaN,19.1,17.9,19.4,...,92,93,122,1.8,0.1,SAO PAULO - MIRANTE,A701,SP,"-23,49638888","-46,61999999"
2,2022/01/01,0200 UTC,0.0,920.8,921.0,920.7,NaN,19.0,17.9,19.1,...,93,93,19,1.8,0.0,SAO PAULO - MIRANTE,A701,SP,"-23,49638888","-46,61999999"


In [5]:
# .to_string() forces it to print in the output cell as plain text
# easier to read and copy than the rendered table
print(df_sample.head(3).to_string())

         Data  Hora UTC  PRECIPITAÇÃO TOTAL, HORÁRIO (mm)  PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)  PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)  PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)  RADIACAO GLOBAL (Kj/m²)  TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)  TEMPERATURA DO PONTO DE ORVALHO (°C)  TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)  TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)  TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)  TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)  UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)  UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)  UMIDADE RELATIVA DO AR, HORARIA (%)  VENTO, DIREÇÃO HORARIA (gr) (° (gr))  VENTO, RAJADA MAXIMA (m/s)  VENTO, VELOCIDADE HORARIA (m/s)              estacao codigo_wmo  uf      latitude     longitude
0  2022/01/01  0000 UTC                               0.0                                                  920.5                                            920.5                                             9

* RADIACAO GLOBAL is NaN for the first 3 rows — nighttime readings, expected
* Hora UTC is 0000 UTC, 0100 UTC etc — needs parsing into proper datetime
* latitude and longitude are strings with comma decimals (-23,49638888) — need converting to float
* Numeric columns are already parsed as floats correctly — the decimal="," parameter worked

In [6]:
# Glob all CSV files recursively under data/raw/INMET
all_files = sorted((RAW / "INMET").glob("**/*.CSV"))

print(f"Total files found: {len(all_files)}\n")

# Read each file and print a one-line summary
# We're not storing anything yet — just inspecting
for filepath in all_files:
    try:
        df, meta = read_inmet_file(filepath)
        print(
            f"{meta['estacao']:<45}"  # station name, left-aligned in 45 chars
            f"  {filepath.parent.name}"  # year folder
            f"  rows: {len(df)}"
            f"  nulls: {df.isnull().sum().sum()}"  # total null count across all columns
        )
    except Exception as e:
        # If any file fails, we want to know which one and why
        # without stopping the whole loop
        print(f"ERROR in {filepath.name}: {e}")

Total files found: 33

BRASILIA                                       2022  rows: 8760  nulls: 4084
MANAUS                                         2022  rows: 8760  nulls: 1118
PORTO ALEGRE - JARDIM BOTANICO                 2022  rows: 8760  nulls: 4553
PORTO ALEGRE- BELEM NOVO                       2022  rows: 576  nulls: 341
BELO HORIZONTE (PAMPULHA)                      2022  rows: 8760  nulls: 4159
BELO HORIZONTE - CERCADINHO                    2022  rows: 8760  nulls: 1426
SAO PAULO - MIRANTE                            2022  rows: 8760  nulls: 4198
SAO PAULO - INTERLAGOS                         2022  rows: 8760  nulls: 4540
BRASILIA                                       2023  rows: 8760  nulls: 4251
MANAUS                                         2023  rows: 8760  nulls: 2132
PORTO ALEGRE - JARDIM BOTANICO                 2023  rows: 8760  nulls: 5688
PORTO ALEGRE- BELEM NOVO                       2023  rows: 8760  nulls: 32
BELO HORIZONTE (PAMPULHA)                      2023  rows

## Normal / expected

* 8760 rows = 365 days × 24h. 8784 = 366 × 24 (2024 is a leap year). Both correct.
* Porto Alegre Belém Novo 2022 has only 576 rows — started December 8th, roughly 24 days. Matches the filename.
* BH Santo Agostinho 2025 has 3024 rows — started August 28th. Also matches.
* Porto Alegre Belém Novo 2025 has 5880 rows — started May 1st. Matches.
* RADIACAO GLOBAL being null at night explains a large chunk of nulls across all files — roughly 4000 nulls on a full year file is consistent with half the hours being nighttime.*

## Genuinely concerning

* SAO PAULO - INTERLAGOS 2024: 19696 nulls on 8784 rows with 19 columns of data — that's nearly all columns null for most rows. Something went badly wrong with that station that year.
* SAO PAULO - MIRANTE 2023: 8036 nulls. Also very high.
* SAO PAULO - INTERLAGOS 2023: 7782 nulls. Same pattern.
* PORTO ALEGRE - BELEM NOVO 2025: 67686 nulls on 5880 rows — mathematically impossible unless there are more columns than expected. Worth investigating.
* PORTO ALEGRE - JARDIM BOTANICO 2023: 5688 nulls, noticeably worse than other years.

In [7]:
# Inspect the two most problematic files in detail
# We want to know: which columns are null, and how many rows are fully empty

problem_files = [
    RAW / "INMET" / "2024" / "INMET_SE_SP_A771_SAO PAULO - INTERLAGOS_01-01-2024_A_31-12-2024.CSV",
    RAW / "INMET" / "2025" / "INMET_S_RS_B825_PORTO ALEGRE - BELEM NOVO_01-05-2025_A_31-12-2025.CSV",
]

for filepath in problem_files:
    df, meta = read_inmet_file(filepath)
    print(f"\n{'='*60}")
    print(f"{meta['estacao']} — {filepath.parent.name}")
    print(f"Shape: {df.shape}")
    print(f"\nNulls per column:")
    print(df.isnull().sum().to_string())
    print(f"\nFully empty rows: {df.isnull().all(axis=1).sum()}")


SAO PAULO - INTERLAGOS — 2024
Shape: (8784, 24)

Nulls per column:
Data                                                        0
Hora UTC                                                    0
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                            9
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)       8
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)             8
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)            8
RADIACAO GLOBAL (Kj/m²)                                  4065
TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)              984
TEMPERATURA DO PONTO DE ORVALHO (°C)                      982
TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)                982
TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)                982
TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)          982
TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)          982
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  982
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                  982
UM

São Paulo Interlagos 2024 — not as bad as it looked. The ~982 nulls on temperature/humidity columns all land on the same rows, meaning the station was offline for roughly 41 consecutive days (982 ÷ 24 ≈ 41). The rest of the year is fine. This is a station outage, not a data corruption problem. We'll handle it in cleaning with a note, but we're not throwing this file out.
Porto Alegre Belém Novo 2025 — 3981 nulls on almost every column, out of 5880 rows. That's 67% of all readings completely empty. The station started May 1st, so it has ~245 days of data — but only ~99 days are actually usable. This station is borderline useless for 2025. We'll flag it and likely exclude it from analysis, keeping only Jardim Botânico as the Porto Alegre representative for that year.

In [8]:
# Check if the Interlagos 2024 nulls are concentrated in a continuous block
# (station outage) or scattered randomly (sensor degradation)
interlagos_2024 = RAW / "INMET" / "2024" / "INMET_SE_SP_A771_SAO PAULO - INTERLAGOS_01-01-2024_A_31-12-2024.CSV"
df_inter, _ = read_inmet_file(interlagos_2024)

# Find rows where temperature is null — our proxy for "station offline"
# If it's an outage, these will be a contiguous block of dates
null_mask = df_inter["TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)"].isnull()

print(f"First null row: {df_inter.loc[null_mask, 'Data'].iloc[0]}")
print(f"Last null row:  {df_inter.loc[null_mask, 'Data'].iloc[-1]}")
print(f"Total null rows: {null_mask.sum()}")
print(f"\nAre the nulls contiguous (one outage block)?")

# Check if all null rows fall between the first and last null date
# by comparing null count to the row span between them
first_idx = null_mask.idxmax()
last_idx = len(df_inter) - 1 - null_mask[::-1].values.argmax()
span = last_idx - first_idx + 1
print(f"Row span from first to last null: {span}")
print(f"Null rows within that span: {null_mask[first_idx:last_idx+1].sum()}")
print(f"Conclusion: {'contiguous outage' if span == null_mask.sum() else 'scattered nulls — sensor degradation'}")

First null row: 2024/01/04
Last null row:  2024/12/11
Total null rows: 984

Are the nulls contiguous (one outage block)?
Row span from first to last null: 8210
Null rows within that span: 984
Conclusion: scattered nulls — sensor degradation


# =============================================================
# INSPECTION SUMMARY — key findings for notebook 02
# =============================================================

FILES
- 33 CSV files total across 2022–2025
- All full-year files have correct row counts (8760 or 8784 for leap year 2024)

STATIONS TO FLAG
- SAO PAULO - INTERLAGOS 2024
    Scattered nulls on temperature/humidity (~11% of rows) throughout the year
    Cause: sensor degradation. Treatment: interpolate short gaps, leave long ones as NaN

- PORTO ALEGRE - BELEM NOVO 2025
    ~67% of rows fully empty despite station being active from May 1st
    Treatment: exclude from 2025 analysis, use JARDIM BOTANICO only for that year

- SAO PAULO - MIRANTE 2023 and INTERLAGOS 2023
    Higher than usual nulls — inspect in notebook 02 before deciding treatment

KNOWN STRUCTURAL ISSUES FOR CLEANING (notebook 02)
- Column names: long Portuguese strings → rename to snake_case
- Datetime: 'Data' + 'Hora UTC' are separate columns → combine into single timestamp
- Decimals: latitude/longitude read as strings with comma separator → convert to float
- RADIACAO GLOBAL: always null at night → expected, not a data quality issue
- Porto Alegre Belem Novo station code changes B807 → B825 across years → normalize
- BH Santo Agostinho 2025: only starts 28-08-2025 → very partial, decide on inclusion
